# Units

This package provides unit conversion tools 

In [ ]:
from beamphysics.units import pmd_unit, NAMED_UNITS, dimension_name, sqrt_unit
from beamphysics import particle_paths
from h5py import File

This is the basic class:

In [ ]:
help(pmd_unit)

Get a known units. These can be multiplied and divided:

In [ ]:
u1 = pmd_unit("J")
u2 = pmd_unit("m")
u1, u2, u1 / u2, u1 * u2

Special function for `sqrt`. It halves the dimension exponents, so the result can carry fractional dimensions:

In [ ]:
sqrt_unit(u1)

## Power notation

Units can be raised to powers using `^` notation. Integer, decimal, and fractional powers are supported:

Integer and decimal powers:

In [ ]:
pmd_unit("m^2"), pmd_unit("m^-1"), pmd_unit("m^0.5")

Fractional powers using parentheses, e.g. `m^(-3/2)`, `m^(1/2)`:

In [ ]:
pmd_unit("m^(-3/2)"), pmd_unit("m^(3/2)")

`sqrt()` notation is equivalent to `^(1/2)`:

In [ ]:
pmd_unit("sqrt(m)"), pmd_unit("m^(1/2)")

Powers work in compound expressions:

In [ ]:
pmd_unit("eV/c*m^(-1/2)")

Fractional dimensions are preserved:

In [ ]:
u = pmd_unit("m^(-3/2)")
u.unitDimension  # Length dimension is -1.5

accepts `*` and `/`  between two known symbols.

In [ ]:
pmd_unit("W*s")

# Named units
The package knows the following units with these symbol named 

In [ ]:
NAMED_UNITS

# SI prefixes

Any known unit symbol may carry a standard SI prefix (`k`, `M`, `G`, `T`, `m`, `µ`, `n`, ...). The prefix scales `unitSI` and leaves the dimension unchanged:

In [ ]:
pmd_unit("keV"), pmd_unit("mm"), pmd_unit("GHz")

Prefixes also apply to the dimensionless angle units `rad` and `degree` (e.g. milliradians):

In [ ]:
pmd_unit("mrad"), pmd_unit("µrad")

A prefix is only accepted when the remainder is itself a known unit, and a bare prefix is **not** a unit. Symbols that are both a prefix letter and a unit keep their unit meaning — `c` is the speed of light (not centi) and `T` is tesla (not tera):

In [ ]:
pmd_unit("c"), pmd_unit("T")  # the units, not centi-/tera-

A bare prefix, or a prefixed dimensionless identity, is rejected:

In [ ]:
for s in ["k", "M", "k1"]:
    try:
        pmd_unit(s)
    except ValueError as ex:
        print(f"{s!r} -> {type(ex).__name__}: {ex}")

# `.simplify()`

The `.simplify()` method searches the internal `NAMED_UNITS` list for an equivalent unit with a simpler symbol. It can:

- collapse a product or quotient to a single named unit (`W*s` &rarr; `J`, `V*A` &rarr; `W`, `J/C` &rarr; `V`);
- reconstruct an `a*b` / `a/b` compound when no single name exists (e.g. `T*m`, `T/m`);
- attach an SI prefix when the value is an exact power-of-ten multiple of a named unit (`1000 J` &rarr; `kJ`).

Matching is float-tolerant and deterministic (the simplest symbol wins), and the resulting symbol always re-parses back to the same unit. `.simplify()` is not called automatically.

In [ ]:
u = pmd_unit("W*s")
u.simplify()

A product or quotient of named units, or a reconstructed compound:

In [ ]:
(
    (pmd_unit("V") * pmd_unit("A")).simplify(),  # -> W
    (pmd_unit("J") / pmd_unit("C")).simplify(),  # -> V
    (pmd_unit("T") * pmd_unit("m")).simplify(),  # -> T*m (no single name)
)

In [ ]:
u = pmd_unit("this should be kJ", 1e3, (2, 1, -2, 0, 0, 0, 0))
simplified = u.simplify()

# The simplified symbol re-parses back to the same unit:
simplified, pmd_unit(simplified.unitSymbol)

A purely dimensionless quantity has no meaningful named form, so `.simplify()` leaves it unchanged rather than inventing a same-dimension ratio. For example, `mrad/µrad` is just the number `1000`, and it is **not** rewritten as `kg/g`:

In [ ]:
pmd_unit("mrad/µrad").simplify()  # 1000, dimensionless -> unchanged

# openPMD HDF5 units

Open a file, find the particle paths from the root attributes

In [ ]:
# Pick one:
# H5FILE = 'data/bmad_particles.h5'
H5FILE = "data/distgen_particles.h5"
# H5FILE = 'data/astra_particles.h5'
h5 = File(H5FILE, "r")

ppaths = particle_paths(h5)
print(ppaths)

This points to a single particle group:

In [ ]:
ph5 = h5[ppaths[0]]
list(ph5)

Each component should have a dimension and a conversion factor to SI:

In [ ]:
d = dict(ph5["momentum/x"].attrs)
d

In [ ]:
tuple(d["unitDimension"])

This will extract the name of this dimension:

In [ ]:
dimension_name(d["unitDimension"])

# Nice arrays

In [ ]:
from beamphysics.units import nice_array

This will scale the array, and return the appropriate SI prefix:

In [ ]:
x = 1e-4
unit = "m"
nice_array(x)

In [ ]:
nice_array([-0.01, 0.01])

In [ ]:
from beamphysics.units import nice_scale_prefix

In [ ]:
nice_scale_prefix(0.009)

# Limitations

This is a simple class for use with this package. So even simple things like the example below will fail. 

For more advanced units, use a package like Pint: https://pint.readthedocs.io/


In [ ]:
try:
    u1 / 1
except Exception as ex:
    print(f"You can't do this. It will raise {type(ex).__name__}: {ex}")